# Crop Recommendation Model Training (Kaggle Dataset)
This notebook downloads the Kaggle dataset, trains a model, evaluates it, and saves artifacts for backend inference.

## 1) Install Dependencies

In [ ]:
!pip -q install kaggle scikit-learn pandas numpy joblib seaborn

## 2) Upload Kaggle API Key
Download `kaggle.json` from Kaggle (Account -> API) and upload it here.

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json

## 3) Configure Kaggle and Download Dataset

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d atharvaingle/crop-recommendation-dataset -p /content --unzip
!ls -la

## 4) Load Dataset

In [ ]:
import pandas as pd

csv_path = "/content/Crop_recommendation.csv"
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
df.head()

## 5) Quick Data Checks

In [ ]:
df.info()
print("\nMissing values:\n", df.isna().sum())
print("\nLabel distribution:\n", df["label"].value_counts().head())

## 6) Train/Test Split and Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 7) Train Model (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

print("Training complete")

## 8) Evaluate Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, pred)

print("Accuracy:", acc)
print("\nClassification Report:\n", classification_report(y_test, pred))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

labels = sorted(y.unique())
cm = confusion_matrix(y_test, pred, labels=labels)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 9) Save Model and Scaler

In [ ]:
import joblib

joblib.dump(model, "crop_recommendation_model.joblib")
joblib.dump(scaler, "crop_recommendation_scaler.joblib")

print("Saved model and scaler")

## 10) Inference Sanity Check

In [ ]:
sample = X.iloc[0:1]
sample_scaled = scaler.transform(sample)
pred_label = model.predict(sample_scaled)[0]
proba = model.predict_proba(sample_scaled)[0]

print("Sample:", sample.to_dict(orient="records")[0])
print("Prediction:", pred_label)
print("Top confidence:", proba.max())

## 11) Download Artifacts
Run this to download the model and scaler to your local machine.

In [ ]:
from google.colab import files

files.download("crop_recommendation_model.joblib")
files.download("crop_recommendation_scaler.joblib")